# Sistema de Recomendação de Produtos Financeiros
## Case Técnico — Ciência de Dados

**Abordagem:** LightGBM com Learning to Rank (LambdaMART) + Features contextuais  
**Autor:** Candidato  
**Data:** 2025


## 0. Setup e Dependências

In [ ]:
# requirements.txt equivalente:
# pandas>=2.0.0
# numpy>=1.24.0
# lightgbm>=4.0.0
# scikit-learn>=1.3.0
# matplotlib>=3.7.0
# seaborn>=0.12.0
# scipy>=1.11.0

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import os
from pathlib import Path

warnings.filterwarnings('ignore')

# Seeds fixas para reprodutibilidade
SEED = 42
np.random.seed(SEED)

# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

DATA_DIR = Path('data')
print('Setup concluído.')

## 1. Geração de Dados Sintéticos

Como os dados reais não foram fornecidos, geramos dados sintéticos que respeitam as especificações do README, incluindo distribuições realistas e correlações entre variáveis.

In [ ]:
def gerar_dados_sinteticos(n_clientes=50000, seed=42):
    """Gera datasets sintéticos conforme especificação do case."""
    rng = np.random.RandomState(seed)
    os.makedirs('data', exist_ok=True)

    # ── PRODUTOS ────────────────────────────────────────────────
    produtos_info = [
        ('cartao_platinum',      'cartao',        350, 80, 0.77, 'premium'),
        ('cartao_black',         'cartao',        600, 150, 0.75, 'premium'),
        ('credito_pessoal',      'credito',       280, 60, 0.79, 'todos'),
        ('credito_consignado',   'credito',       420, 90, 0.79, 'funcionários_publicos'),
        ('cheque_especial',      'credito',        80, 20, 0.75, 'todos'),
        ('uso_lastro_limite',    'credito',       150, 30, 0.80, 'investidores'),
        ('pix_parcelado',        'credito',       120, 25, 0.79, 'todos'),
        ('investimento_cdb',     'investimento',  200, 40, 0.80, 'todos'),
        ('investimento_lci_lca', 'investimento',  250, 50, 0.80, 'premium_intermediario'),
        ('previdencia',          'investimento',  500, 100, 0.80, 'todos'),
        ('tesouro_direto',       'investimento',  100, 20, 0.80, 'iniciantes'),
        ('seguro_vida',          'seguro',        180, 40, 0.78, 'todos'),
        ('seguro_residencial',   'seguro',        220, 50, 0.77, 'proprietarios'),
        ('seguro_auto',          'seguro',        300, 70, 0.77, 'motoristas'),
        ('seguro_viagem',        'seguro',        150, 30, 0.80, 'viajantes'),
        ('consorcio_auto',       'consorcio',     800, 200, 0.75, 'todos'),
        ('consorcio_imovel',     'consorcio',    1500, 400, 0.73, 'premium'),
        ('conta_digital_plus',   'conta',          50, 10, 0.80, 'todos'),
        ('titulo_capitalizacao', 'capitalizacao',  60, 15, 0.75, 'todos'),
        ('portabilidade_salario','conta',          30, 5,  0.83, 'todos'),
    ]
    df_produtos = pd.DataFrame(
        produtos_info,
        columns=['produto','categoria','receita_media','custo_aquisicao','margem','publico_alvo']
    )
    df_produtos.to_csv('data/produtos.csv', index=False)
    print('produtos.csv gerado.')

    # ── CLIENTES ────────────────────────────────────────────────
    n = n_clientes
    ids = np.arange(1, n+1)
    segmentos = rng.choice(['basico','intermediario','premium'], size=n, p=[0.50, 0.35, 0.15])
    seg_num = np.where(segmentos=='basico', 0, np.where(segmentos=='intermediario', 1, 2))

    renda_base = np.where(seg_num==0,
                          rng.lognormal(7.8, 0.4, n),
                 np.where(seg_num==1,
                          rng.lognormal(8.8, 0.5, n),
                          rng.lognormal(10.0, 0.6, n)))

    score_base = 300 + seg_num*200 + rng.normal(0, 80, n)
    score_base = np.clip(score_base, 0, 1000)

    ufs = ['SP','RJ','MG','BA','RS','PR','PE','CE','GO','SC',
           'MA','PB','AM','ES','RN','MT','MS','PI','AL','RO',
           'TO','AC','AP','RR','SE','DF','PA']
    uf_probs = np.array([0.22,0.12,0.10,0.07,0.06,0.06,0.05,0.04,
                         0.03,0.04,0.02,0.02,0.02,0.02,0.02,0.02,
                         0.02,0.01,0.01,0.01,0.01,0.01,0.01,0.01,
                         0.01,0.03,0.02])
    uf_probs = uf_probs / uf_probs.sum()

    idade = rng.randint(18, 81, n)
    saldo_medio = renda_base * rng.uniform(0.3, 3.0, n)
    inv = np.where(seg_num==0,
                   rng.exponential(500, n),
          np.where(seg_num==1,
                   rng.exponential(5000, n),
                   rng.exponential(50000, n)))

    limite = renda_base * rng.uniform(0.5, 4.0, n)
    pct_util = rng.beta(2, 5, n)

    df_clientes = pd.DataFrame({
        'id_cliente': ids,
        'idade': idade,
        'genero': rng.choice(['M','F','Outro'], size=n, p=[0.48,0.49,0.03]),
        'uf': rng.choice(ufs, size=n, p=uf_probs),
        'segmento': segmentos,
        'score_credito': np.round(score_base, 1),
        'renda_mensal': np.round(renda_base, 2),
        'saldo_medio_conta': np.round(saldo_medio, 2),
        'qtd_meses_cliente': rng.randint(1, 121, n),
        'qtd_produtos_ativos': rng.randint(0, 10, n),
        'qtd_transacoes_pix_6m': rng.poisson(20 + seg_num*15, n),
        'vlr_total_investimentos': np.round(inv, 2),
        'vlr_medio_gasto_cartao': np.round(renda_base * rng.uniform(0.05, 0.4, n), 2),
        'vlr_medio_gasto_alimentacao': np.round(renda_base * rng.uniform(0.08, 0.20, n), 2),
        'vlr_medio_gasto_transporte': np.round(renda_base * rng.uniform(0.03, 0.12, n), 2),
        'vlr_medio_gasto_saude': np.round(renda_base * rng.uniform(0.02, 0.10, n), 2),
        'vlr_medio_gasto_educacao': np.round(renda_base * rng.uniform(0.01, 0.08, n), 2),
        'vlr_medio_gasto_lazer': np.round(renda_base * rng.uniform(0.01, 0.10, n), 2),
        'ind_debito_automatico': rng.randint(0, 2, n),
        'qtd_dias_inatividade': rng.exponential(15, n).astype(int),
        'vlr_limite_credito': np.round(limite, 2),
        'pct_utilizacao_limite': np.round(pct_util, 4),
        'canal_preferencial': rng.choice(['app','web','agencia','telefone'], size=n, p=[0.55,0.25,0.12,0.08]),
    })
    df_clientes.to_csv('data/clientes.csv', index=False)
    print('clientes.csv gerado.')

    # ── INTERAÇÕES ──────────────────────────────────────────────
    produtos_list = [p[0] for p in produtos_info]
    canais_int = ['app_home','push_notification','app_busca','email','web_banking']
    safras = [f'2024{m:02d}' for m in range(1,13)] + [f'2025{m:02d}' for m in range(1,13)]

    # Probabilidade de contratação por produto (popularidade)
    pop_prod = np.array([0.04,0.02,0.08,0.03,0.10,0.02,0.06,
                         0.07,0.04,0.03,0.05,0.05,0.03,0.04,
                         0.02,0.02,0.01,0.09,0.06,0.09])

    records = []
    id_int = 1
    n_int = 500000

    # Amostra de clientes para as interações
    clientes_sample = rng.choice(ids, size=n_int, replace=True)
    produtos_sample = rng.choice(len(produtos_list), size=n_int, p=pop_prod/pop_prod.sum())
    safras_sample = rng.choice(safras, size=n_int)
    canais_sample = rng.choice(canais_int, size=n_int, p=[0.50,0.20,0.15,0.10,0.05])
    posicoes = rng.randint(1, 21, n_int)

    # Bias de posição: posições menores têm mais cliques
    pos_bias = 1.0 / np.sqrt(posicoes)

    # Probabilidade de clique com bias de posição
    p_clique_base = np.array([pop_prod[p] for p in produtos_sample])
    p_clique = np.clip(p_clique_base * pos_bias * 3, 0, 0.8)
    clicou = (rng.uniform(size=n_int) < p_clique).astype(int)

    # Conversão apenas para quem clicou
    p_conv = 0.25
    contratou = np.where(clicou == 1,
                         (rng.uniform(size=n_int) < p_conv).astype(int),
                         0)

    receitas_por_prod = np.array([p[2] for p in produtos_info])
    receita = np.where(contratou == 1,
                       receitas_por_prod[produtos_sample] * rng.uniform(0.8, 1.2, n_int),
                       0.0)

    # Timestamps realistas dentro de cada safra
    timestamps = []
    for s in safras_sample:
        ano, mes = int(s[:4]), int(s[4:])
        import calendar
        _, last_day = calendar.monthrange(ano, mes)
        ts = pd.Timestamp(year=ano, month=mes, day=1) + \
             pd.to_timedelta(rng.randint(0, last_day*24*3600), unit='s')
        timestamps.append(ts)

    df_int = pd.DataFrame({
        'id_interacao': np.arange(1, n_int+1),
        'id_cliente': clientes_sample,
        'produto': [produtos_list[p] for p in produtos_sample],
        'posicao_exibicao': posicoes,
        'canal': canais_sample,
        'clicou': clicou,
        'contratou': contratou,
        'receita_gerada': np.round(receita, 2),
        'timestamp': timestamps,
        'safra': safras_sample,
    })
    df_int.to_csv('data/interacoes.csv', index=False)
    print('interacoes.csv gerado.')

    # ── CONTRATOS ATIVOS ────────────────────────────────────────
    n_contratos = 62000
    cli_contr = rng.choice(ids, size=n_contratos, replace=True)
    prod_contr = rng.choice(produtos_list, size=n_contratos)
    status_contr = rng.choice(['ativo','cancelado','vencido'], size=n_contratos, p=[0.70,0.15,0.15])

    datas = pd.to_datetime('2022-01-01') + \
            pd.to_timedelta(rng.randint(0, 365*3, n_contratos), unit='D')

    df_contr = pd.DataFrame({
        'id_cliente': cli_contr,
        'produto': prod_contr,
        'data_contratacao': datas.strftime('%Y-%m-%d'),
        'status': status_contr,
        'canal_contratacao': rng.choice(['app','web','agencia','telemarketing'], size=n_contratos, p=[0.50,0.25,0.15,0.10]),
    })
    # Remove duplicatas (mesmo cliente + mesmo produto ativo)
    df_contr = df_contr.drop_duplicates(subset=['id_cliente','produto'])
    df_contr.to_csv('data/contratos_ativos.csv', index=False)
    print('contratos_ativos.csv gerado.')

    return df_clientes, df_int, df_produtos, df_contr


# Verifica se já existem os CSVs; se não, gera
if not Path('data/clientes.csv').exists():
    print('Gerando dados sintéticos...')
    df_clientes, df_interacoes, df_produtos, df_contratos = gerar_dados_sinteticos()
else:
    print('Carregando dados existentes...')
    df_clientes = pd.read_csv('data/clientes.csv')
    df_interacoes = pd.read_csv('data/interacoes.csv')
    df_produtos = pd.read_csv('data/produtos.csv')
    df_contratos = pd.read_csv('data/contratos_ativos.csv')

print(f'\nShapes:')
print(f'  clientes:   {df_clientes.shape}')
print(f'  interacoes: {df_interacoes.shape}')
print(f'  produtos:   {df_produtos.shape}')
print(f'  contratos:  {df_contratos.shape}')

## 2. Análise Exploratória dos Dados (EDA)

In [ ]:
# ── 2.1 Visão geral dos dados ──────────────────────────────────
print('=== CLIENTES ===')
print(df_clientes.dtypes)
print('\nMissing values:')
print(df_clientes.isnull().sum())
print('\nEstatísticas descritivas (numéricas):')
df_clientes.describe()

In [ ]:
# ── 2.2 Distribuição por segmento ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribuições dos Clientes por Segmento', fontsize=15, fontweight='bold')

# Contagem por segmento
seg_counts = df_clientes['segmento'].value_counts()
axes[0,0].bar(seg_counts.index, seg_counts.values, color=['#2196F3','#FF9800','#4CAF50'])
axes[0,0].set_title('Volume por Segmento')
axes[0,0].set_ylabel('Quantidade')
for i, v in enumerate(seg_counts.values):
    axes[0,0].text(i, v + 200, f'{v:,}', ha='center', fontsize=10)

# Renda por segmento
df_clientes.boxplot(column='renda_mensal', by='segmento', ax=axes[0,1])
axes[0,1].set_title('Renda Mensal por Segmento')
axes[0,1].set_xlabel('')
axes[0,1].set_yscale('log')
plt.sca(axes[0,1])
plt.title('Renda Mensal por Segmento')

# Score de crédito por segmento
for seg, color in zip(['basico','intermediario','premium'], ['#2196F3','#FF9800','#4CAF50']):
    subset = df_clientes[df_clientes['segmento']==seg]['score_credito']
    axes[0,2].hist(subset, bins=30, alpha=0.6, label=seg, color=color)
axes[0,2].set_title('Distribuição de Score de Crédito')
axes[0,2].legend()
axes[0,2].set_xlabel('Score')

# Distribuição de idade
axes[1,0].hist(df_clientes['idade'], bins=30, color='#9C27B0', edgecolor='white')
axes[1,0].set_title('Distribuição de Idade')
axes[1,0].set_xlabel('Idade')
axes[1,0].set_ylabel('Frequência')

# Canal preferencial
canal_counts = df_clientes['canal_preferencial'].value_counts()
axes[1,1].pie(canal_counts.values, labels=canal_counts.index, autopct='%1.1f%%',
              colors=['#FF5722','#607D8B','#795548','#009688'])
axes[1,1].set_title('Canal Preferencial')

# Investimentos por segmento (escala log)
df_clientes[df_clientes['vlr_total_investimentos'] > 0].boxplot(
    column='vlr_total_investimentos', by='segmento', ax=axes[1,2]
)
axes[1,2].set_yscale('log')
axes[1,2].set_title('Volume de Investimentos por Segmento (log)')
axes[1,2].set_xlabel('')
plt.sca(axes[1,2])
plt.title('Investimentos por Segmento (log)')

plt.suptitle('Distribuições dos Clientes por Segmento', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_clientes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.3 Análise de Interações ─────────────────────────────────
df_interacoes['timestamp'] = pd.to_datetime(df_interacoes['timestamp'])

print('=== INTERAÇÕES ===')
print(f'Total: {len(df_interacoes):,}')
print(f'Taxa de clique geral: {df_interacoes["clicou"].mean():.2%}')
print(f'Taxa de contratação geral: {df_interacoes["contratou"].mean():.2%}')
print(f'Taxa de conversão (click→contract): {df_interacoes[df_interacoes["clicou"]==1]["contratou"].mean():.2%}')
print(f'Receita total gerada: R$ {df_interacoes["receita_gerada"].sum():,.2f}')

In [ ]:
# ── 2.4 Taxa de contratação por produto ──────────────────────
prod_stats = df_interacoes.groupby('produto').agg(
    impressoes=('id_interacao','count'),
    cliques=('clicou','sum'),
    contratos=('contratou','sum'),
    receita_total=('receita_gerada','sum'),
).reset_index()
prod_stats['ctr'] = prod_stats['cliques'] / prod_stats['impressoes']
prod_stats['taxa_contratacao'] = prod_stats['contratos'] / prod_stats['impressoes']
prod_stats['receita_media_imp'] = prod_stats['receita_total'] / prod_stats['impressoes']
prod_stats = prod_stats.sort_values('taxa_contratacao', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Taxa de contratação por produto
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(prod_stats)))
bars = axes[0].barh(prod_stats['produto'], prod_stats['taxa_contratacao'], color=colors)
axes[0].set_title('Taxa de Contratação por Produto', fontweight='bold')
axes[0].set_xlabel('Taxa de Contratação')
axes[0].invert_yaxis()

# Receita média por impressão
prod_stats2 = prod_stats.sort_values('receita_media_imp', ascending=False)
bars2 = axes[1].barh(prod_stats2['produto'], prod_stats2['receita_media_imp'],
                     color=plt.cm.Blues(np.linspace(0.4, 0.9, len(prod_stats2))))
axes[1].set_title('Receita Média por Impressão (R$)', fontweight='bold')
axes[1].set_xlabel('Receita Média (R$)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('eda_produtos.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 por taxa de contratação:')
print(prod_stats[['produto','impressoes','contratos','taxa_contratacao','receita_total']].head())

In [ ]:
# ── 2.5 Bias de posição no carrossel ─────────────────────────
pos_stats = df_interacoes.groupby('posicao_exibicao').agg(
    impressoes=('id_interacao','count'),
    ctr=('clicou','mean'),
    taxa_contratacao=('contratou','mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(pos_stats['posicao_exibicao'], pos_stats['ctr'], 'o-', color='#2196F3', linewidth=2)
axes[0].axvline(x=5.5, color='red', linestyle='--', label='Limite visível sem scroll')
axes[0].set_title('CTR por Posição no Carrossel', fontweight='bold')
axes[0].set_xlabel('Posição')
axes[0].set_ylabel('CTR')
axes[0].legend()

axes[1].plot(pos_stats['posicao_exibicao'], pos_stats['taxa_contratacao'], 's-', color='#4CAF50', linewidth=2)
axes[1].axvline(x=5.5, color='red', linestyle='--', label='Limite visível sem scroll')
axes[1].set_title('Taxa de Contratação por Posição', fontweight='bold')
axes[1].set_xlabel('Posição')
axes[1].set_ylabel('Taxa de Contratação')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_posicao.png', dpi=150, bbox_inches='tight')
plt.show()

print('Queda de CTR após posição 5:')
top5 = pos_stats[pos_stats['posicao_exibicao'] <= 5]['ctr'].mean()
rest = pos_stats[pos_stats['posicao_exibicao'] > 5]['ctr'].mean()
print(f'  Posições 1-5:  {top5:.4f}')
print(f'  Posições 6-20: {rest:.4f}')
print(f'  Razão: {top5/rest:.1f}x')

In [ ]:
# ── 2.6 Análise temporal (safras) ─────────────────────────────
safra_stats = df_interacoes.groupby('safra').agg(
    interacoes=('id_interacao','count'),
    contratos=('contratou','sum'),
    taxa_contratacao=('contratou','mean'),
    receita=('receita_gerada','sum'),
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Análise Temporal por Safra', fontsize=14, fontweight='bold')

x = range(len(safra_stats))
axes[0,0].bar(x, safra_stats['interacoes'], color='#3F51B5')
axes[0,0].set_title('Volume de Interações')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(safra_stats['safra'], rotation=45)

axes[0,1].bar(x, safra_stats['contratos'], color='#4CAF50')
axes[0,1].set_title('Volume de Contratos')
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels(safra_stats['safra'], rotation=45)

axes[1,0].plot(x, safra_stats['taxa_contratacao'], 'o-', color='#FF5722', linewidth=2)
axes[1,0].set_title('Taxa de Contratação')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(safra_stats['safra'], rotation=45)

axes[1,1].bar(x, safra_stats['receita']/1e6, color='#9C27B0')
axes[1,1].set_title('Receita Total (R$ milhões)')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(safra_stats['safra'], rotation=45)

plt.tight_layout()
plt.savefig('eda_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.7 Análise de co-contratação ────────────────────────────
# Produtos com status ativo por cliente
ativos = df_contratos[df_contratos['status'] == 'ativo']

# Matriz de co-ocorrência
produtos_list = df_produtos['produto'].tolist()
n_prod = len(produtos_list)
prod_idx = {p: i for i, p in enumerate(produtos_list)}

coocur = np.zeros((n_prod, n_prod))
for _, group in ativos.groupby('id_cliente'):
    prods = [p for p in group['produto'].values if p in prod_idx]
    for i in range(len(prods)):
        for j in range(len(prods)):
            if i != j:
                coocur[prod_idx[prods[i]], prod_idx[prods[j]]] += 1

# Normalizar por contagens individuais
prod_counts = np.array([ativos[ativos['produto']==p].shape[0] for p in produtos_list])
coocur_norm = coocur / (prod_counts[:, None] + 1e-9)

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(coocur_norm, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(n_prod))
ax.set_yticks(range(n_prod))
ax.set_xticklabels(produtos_list, rotation=90, fontsize=8)
ax.set_yticklabels(produtos_list, fontsize=8)
plt.colorbar(im, ax=ax, label='Taxa de Co-contratação')
ax.set_title('Matriz de Co-contratação de Produtos (Normalizada)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('eda_cocontratacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.8 Análise por canal ──────────────────────────────────────
canal_stats = df_interacoes.groupby('canal').agg(
    interacoes=('id_interacao','count'),
    taxa_clique=('clicou','mean'),
    taxa_contratacao=('contratou','mean'),
).reset_index().sort_values('taxa_contratacao', ascending=False)

print('Taxa de contratação por canal de interação:')
print(canal_stats.to_string(index=False))

## 3. Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder

def build_features(df_clientes, df_interacoes, df_produtos, df_contratos):
    """
    Constrói o dataset de treinamento para o modelo LTR.
    Cada linha = (cliente, produto) com features e label de contratação.
    """
    print('Construindo features...')

    # ── Features de cliente ──────────────────────────────────────
    cli = df_clientes.copy()

    # Encoding de variáveis categóricas
    for col in ['segmento','canal_preferencial','genero']:
        le = LabelEncoder()
        cli[f'{col}_enc'] = le.fit_transform(cli[col].astype(str))

    # UF encoding por frequência
    uf_freq = cli['uf'].value_counts(normalize=True)
    cli['uf_freq'] = cli['uf'].map(uf_freq)

    # Features derivadas
    cli['renda_log'] = np.log1p(cli['renda_mensal'])
    cli['saldo_log'] = np.log1p(cli['saldo_medio_conta'])
    cli['inv_log'] = np.log1p(cli['vlr_total_investimentos'])
    cli['gasto_cartao_log'] = np.log1p(cli['vlr_medio_gasto_cartao'])
    cli['limite_log'] = np.log1p(cli['vlr_limite_credito'])
    cli['renda_saldo_ratio'] = cli['saldo_medio_conta'] / (cli['renda_mensal'] + 1)
    cli['investimento_renda_ratio'] = cli['vlr_total_investimentos'] / (cli['renda_mensal'] + 1)
    cli['gasto_total'] = (cli['vlr_medio_gasto_alimentacao'] +
                          cli['vlr_medio_gasto_transporte'] +
                          cli['vlr_medio_gasto_saude'] +
                          cli['vlr_medio_gasto_educacao'] +
                          cli['vlr_medio_gasto_lazer'])
    cli['propensao_credito'] = (
        (cli['pct_utilizacao_limite'] > 0.7).astype(int) +
        (cli['score_credito'] > 500).astype(int) +
        (cli['renda_mensal'] > 3000).astype(int)
    )
    cli['propensao_investimento'] = (
        (cli['vlr_total_investimentos'] > 1000).astype(int) +
        (cli['saldo_medio_conta'] > 5000).astype(int) +
        (cli['renda_mensal'] > 5000).astype(int)
    )
    cli['faixa_idade'] = pd.cut(cli['idade'], bins=[17,25,35,45,55,65,80],
                                 labels=[0,1,2,3,4,5]).astype(int)

    # ── Produtos já ativos por cliente (para exclusão) ───────────
    ativos_por_cliente = df_contratos[
        df_contratos['status'] == 'ativo'
    ].groupby('id_cliente')['produto'].apply(set).to_dict()

    # ── Histórico de interações por (cliente, produto) ───────────
    hist = df_interacoes.groupby(['id_cliente','produto']).agg(
        n_impressoes=('id_interacao','count'),
        n_cliques=('clicou','sum'),
        n_contratos=('contratou','sum'),
        receita_hist=('receita_gerada','sum'),
        pos_media=('posicao_exibicao','mean'),
        pos_min=('posicao_exibicao','min'),
        ultima_interacao=('timestamp','max'),
    ).reset_index()
    hist['ctr_hist'] = hist['n_cliques'] / (hist['n_impressoes'] + 1)
    hist['conv_hist'] = hist['n_contratos'] / (hist['n_impressoes'] + 1)

    # ── Popularidade global de produtos ─────────────────────────
    pop = df_interacoes.groupby('produto').agg(
        pop_impressoes=('id_interacao','count'),
        pop_ctr=('clicou','mean'),
        pop_conv=('contratou','mean'),
        pop_receita_media=('receita_gerada','mean'),
    ).reset_index()

    # ── Popularidade por segmento ────────────────────────────────
    # join interações com segmento do cliente
    int_seg = df_interacoes.merge(
        df_clientes[['id_cliente','segmento']], on='id_cliente', how='left'
    )
    pop_seg = int_seg.groupby(['segmento','produto']).agg(
        seg_conv=('contratou','mean'),
        seg_ctr=('clicou','mean'),
    ).reset_index()

    return cli, hist, pop, pop_seg, ativos_por_cliente


cli_feat, hist_feat, pop_feat, pop_seg_feat, ativos_por_cliente = build_features(
    df_clientes, df_interacoes, df_produtos, df_contratos
)
print('Features construídas com sucesso.')

## 4. Split Temporal e Construção do Dataset

**Justificativa do corte:** Usamos as 3 últimas safras (2025-10, 2025-11, 2025-12) como teste (~12.5% dos dados). Esse intervalo é suficiente para avaliar com robustez estatística (>60k registros de teste) mantendo volume de treino adequado para o modelo.

In [ ]:
# Split temporal
SAFRAS_TESTE = ['202510', '202511', '202512']
SAFRAS_TREINO = [s for s in df_interacoes['safra'].unique() if s not in SAFRAS_TESTE]

int_treino = df_interacoes[df_interacoes['safra'].isin(SAFRAS_TREINO)]
int_teste = df_interacoes[df_interacoes['safra'].isin(SAFRAS_TESTE)]

print(f'Safras de treino: {sorted(SAFRAS_TREINO)}')
print(f'Safras de teste:  {sorted(SAFRAS_TESTE)}')
print(f'\nRegistros treino: {len(int_treino):,} ({len(int_treino)/len(df_interacoes):.1%})')
print(f'Registros teste:  {len(int_teste):,} ({len(int_teste)/len(df_interacoes):.1%})')

In [ ]:
def build_training_dataset(int_df, cli_feat, hist_feat, pop_feat, pop_seg_feat, df_produtos):
    """
    Constrói o dataset final (cliente x produto) com todas as features.
    Label: contratou (0/1)
    """
    # Cada interação é um exemplo de treino
    df = int_df[['id_cliente','produto','posicao_exibicao','canal','clicou','contratou']].copy()

    # Merge com features do cliente
    cli_cols = ['id_cliente','segmento_enc','canal_preferencial_enc','genero_enc',
                'uf_freq','renda_log','saldo_log','inv_log','gasto_cartao_log',
                'limite_log','renda_saldo_ratio','investimento_renda_ratio',
                'gasto_total','propensao_credito','propensao_investimento',
                'faixa_idade','score_credito','qtd_meses_cliente',
                'qtd_produtos_ativos','qtd_transacoes_pix_6m','ind_debito_automatico',
                'qtd_dias_inatividade','pct_utilizacao_limite','segmento']
    df = df.merge(cli_feat[cli_cols], on='id_cliente', how='left')

    # Merge com histórico do par (cliente, produto)
    df = df.merge(hist_feat, on=['id_cliente','produto'], how='left')
    hist_fill_cols = ['n_impressoes','n_cliques','n_contratos','receita_hist',
                      'pos_media','pos_min','ctr_hist','conv_hist']
    df[hist_fill_cols] = df[hist_fill_cols].fillna(0)

    # Merge com popularidade global
    df = df.merge(pop_feat, on='produto', how='left')

    # Merge com popularidade por segmento
    df = df.merge(pop_seg_feat, on=['segmento','produto'], how='left')

    # Merge com metadados do produto
    prod_cols = ['produto','categoria','receita_media','custo_aquisicao','margem']
    df = df.merge(df_produtos[prod_cols], on='produto', how='left')

    # Encoding de categoria do produto
    le_cat = LabelEncoder()
    df['categoria_enc'] = le_cat.fit_transform(df['categoria'].astype(str))

    # Encoding de canal da interação
    le_canal = LabelEncoder()
    df['canal_enc'] = le_canal.fit_transform(df['canal'].astype(str))

    # Bias de posição como feature
    df['pos_bias'] = 1.0 / np.sqrt(df['posicao_exibicao'])
    df['is_top5'] = (df['posicao_exibicao'] <= 5).astype(int)

    return df, le_cat, le_canal


df_train, le_cat, le_canal = build_training_dataset(
    int_treino, cli_feat, hist_feat, pop_feat, pop_seg_feat, df_produtos
)
df_test, _, _ = build_training_dataset(
    int_teste, cli_feat, hist_feat, pop_feat, pop_seg_feat, df_produtos
)

print(f'Dataset treino: {df_train.shape}')
print(f'Dataset teste:  {df_test.shape}')
print(f'Taxa de contratação treino: {df_train["contratou"].mean():.4f}')
print(f'Taxa de contratação teste:  {df_test["contratou"].mean():.4f}')

## 5. Modelagem — LightGBM (LambdaMART / LTR)

**Justificativa da escolha:**
- **LambdaMART** é um algoritmo de Learning to Rank que otimiza diretamente métricas de ranking (NDCG), sendo ideal para o problema de ordenação do carrossel
- **LightGBM** é eficiente em dados tabulares grandes, suporta variáveis categóricas nativamente e tem boa interpretabilidade via feature importance
- **Alternativa considerada:** Filtragem colaborativa (SVD) — descartada pois não aproveita as features ricas de perfil do cliente
- **Vantagem chave:** A função objetivo de ranking considera explicitamente a posição, o que é crítico dado que apenas as 5 primeiras posições são visíveis

In [ ]:
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

# Features para o modelo
FEATURE_COLS = [
    # Cliente
    'segmento_enc', 'canal_preferencial_enc', 'genero_enc', 'uf_freq',
    'renda_log', 'saldo_log', 'inv_log', 'gasto_cartao_log', 'limite_log',
    'renda_saldo_ratio', 'investimento_renda_ratio', 'gasto_total',
    'propensao_credito', 'propensao_investimento', 'faixa_idade',
    'score_credito', 'qtd_meses_cliente', 'qtd_produtos_ativos',
    'qtd_transacoes_pix_6m', 'ind_debito_automatico', 'qtd_dias_inatividade',
    'pct_utilizacao_limite',
    # Histórico (cliente x produto)
    'n_impressoes', 'n_cliques', 'n_contratos', 'ctr_hist', 'conv_hist',
    'pos_media', 'pos_min', 'receita_hist',
    # Popularidade do produto
    'pop_ctr', 'pop_conv', 'pop_receita_media',
    # Popularidade por segmento
    'seg_conv', 'seg_ctr',
    # Metadados do produto
    'categoria_enc', 'receita_media', 'margem',
    # Contexto
    'canal_enc', 'pos_bias', 'is_top5',
]

# Garantir que todas as colunas existam
for col in FEATURE_COLS:
    if col not in df_train.columns:
        df_train[col] = 0
        df_test[col] = 0

X_train = df_train[FEATURE_COLS].fillna(0)
y_train = df_train['contratou'].values
X_test = df_test[FEATURE_COLS].fillna(0)
y_test = df_test['contratou'].values

# Groups para LTR: número de produtos por consulta (cliente)
# Para simplificação, usamos o número médio de interações por cliente
# Em produção, os groups seriam exatamente os produtos elegíveis por cliente
train_groups = df_train.groupby('id_cliente').size().values
test_groups = df_test.groupby('id_cliente').size().values

print(f'Features: {len(FEATURE_COLS)}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Grupos treino: {len(train_groups)}, médio={train_groups.mean():.1f}')

In [ ]:
# ── Treinamento do modelo LTR ─────────────────────────────────
# Para LambdaMART, usamos objective='lambdarank' no LightGBM

train_dataset = lgb.Dataset(
    X_train, label=y_train,
    group=train_groups,
    feature_name=FEATURE_COLS,
)
test_dataset = lgb.Dataset(
    X_test, label=y_test,
    group=test_groups,
    reference=train_dataset,
    feature_name=FEATURE_COLS,
)

params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [5, 10],
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': 7,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l2': 0.1,
    'min_gain_to_split': 0.01,
    'label_gain': [0, 1],  # ganho para labels 0 e 1
    'verbose': -1,
    'seed': SEED,
    'n_jobs': -1,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=30, verbose=False),
    lgb.log_evaluation(period=50),
]

print('Treinando modelo LambdaMART...')
model = lgb.train(
    params,
    train_dataset,
    num_boost_round=500,
    valid_sets=[test_dataset],
    callbacks=callbacks,
)
print(f'\nMelhor iteração: {model.best_iteration}')

In [ ]:
# ── Feature Importance ────────────────────────────────────────
fi = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance_gain': model.feature_importance(importance_type='gain'),
    'importance_split': model.feature_importance(importance_type='split'),
}).sort_values('importance_gain', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top_n = 20
top_fi = fi.head(top_n)

axes[0].barh(top_fi['feature'][::-1], top_fi['importance_gain'][::-1],
             color=plt.cm.viridis(np.linspace(0.3, 0.9, top_n)))
axes[0].set_title(f'Top {top_n} Features — Gain', fontweight='bold')
axes[0].set_xlabel('Importance (Gain)')

top_fi2 = fi.sort_values('importance_split', ascending=False).head(top_n)
axes[1].barh(top_fi2['feature'][::-1], top_fi2['importance_split'][::-1],
             color=plt.cm.plasma(np.linspace(0.3, 0.9, top_n)))
axes[1].set_title(f'Top {top_n} Features — Split Count', fontweight='bold')
axes[1].set_xlabel('Importance (Split)')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features por Gain:')
print(fi[['feature','importance_gain']].head(10).to_string(index=False))

## 6. Avaliação — Métricas de Recomendação

In [ ]:
def precision_at_k(recommended, relevant, k):
    """Precision@K: fração dos top-K recomendados que são relevantes."""
    top_k = recommended[:k]
    return len(set(top_k) & set(relevant)) / k

def recall_at_k(recommended, relevant, k):
    """Recall@K: fração dos relevantes que aparecem no top-K."""
    if not relevant:
        return 0.0
    top_k = recommended[:k]
    return len(set(top_k) & set(relevant)) / len(relevant)

def hit_rate_at_k(recommended, relevant, k):
    """Hit Rate@K: 1 se ao menos 1 relevante está no top-K."""
    top_k = recommended[:k]
    return 1 if set(top_k) & set(relevant) else 0

def ndcg_at_k(recommended, relevant_scores, k):
    """
    NDCG@K: Normalized Discounted Cumulative Gain.
    relevant_scores: dict {produto: score_relevancia}
    """
    top_k = recommended[:k]
    dcg = sum(
        relevant_scores.get(p, 0) / np.log2(i + 2)
        for i, p in enumerate(top_k)
    )
    # IDCG: ranking ideal
    ideal = sorted(relevant_scores.values(), reverse=True)[:k]
    idcg = sum(score / np.log2(i + 2) for i, score in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def average_precision_at_k(recommended, relevant, k):
    """Average Precision@K."""
    top_k = recommended[:k]
    hits = 0
    ap = 0.0
    for i, p in enumerate(top_k):
        if p in relevant:
            hits += 1
            ap += hits / (i + 1)
    return ap / min(len(relevant), k) if relevant else 0.0


def evaluate_model(model, df_test, X_test, FEATURE_COLS, K_values=[5, 10], label='Modelo LTR'):
    """
    Avalia o modelo calculando métricas de ranking por cliente.
    """
    # Scores do modelo
    df_eval = df_test[['id_cliente','produto','contratou']].copy()
    df_eval['score'] = model.predict(X_test)

    metrics_list = []
    for cliente_id, grp in df_eval.groupby('id_cliente'):
        # Produtos contratados = relevantes
        relevantes = set(grp[grp['contratou']==1]['produto'].values)
        if not relevantes:
            continue
        # Ranking pelo score
        ranking = grp.sort_values('score', ascending=False)['produto'].tolist()
        rel_scores = {p: 1 for p in relevantes}

        row = {'id_cliente': cliente_id}
        for k in K_values:
            row[f'precision@{k}'] = precision_at_k(ranking, relevantes, k)
            row[f'recall@{k}'] = recall_at_k(ranking, relevantes, k)
            row[f'hit_rate@{k}'] = hit_rate_at_k(ranking, relevantes, k)
            row[f'ndcg@{k}'] = ndcg_at_k(ranking, rel_scores, k)
            row[f'map@{k}'] = average_precision_at_k(ranking, relevantes, k)
        metrics_list.append(row)

    df_metrics = pd.DataFrame(metrics_list)
    print(f'\n=== {label} ===')
    print(f'Clientes avaliados: {len(df_metrics):,}')
    for k in K_values:
        print(f'\n  @{k}:')
        for m in ['precision','recall','hit_rate','ndcg','map']:
            col = f'{m}@{k}'
            print(f'    {col:<15}: {df_metrics[col].mean():.4f}')
    return df_metrics


print('Calculando métricas do modelo...')
metrics_model = evaluate_model(model, df_test, X_test, FEATURE_COLS, label='LambdaMART (LightGBM)')

In [ ]:
# ── Baselines ─────────────────────────────────────────────────

class PopularityBaseline:
    """Recomenda os produtos mais populares (maior taxa de contratação)."""
    def __init__(self):
        self.ranking = None

    def fit(self, df_interacoes):
        pop = df_interacoes.groupby('produto')['contratou'].mean().sort_values(ascending=False)
        self.ranking = pop.index.tolist()
        self.scores = pop.values
        return self

    def predict(self, produtos_candidatos):
        # Retorna os candidatos na ordem de popularidade global
        ordered = [p for p in self.ranking if p in set(produtos_candidatos)]
        return ordered


def evaluate_baseline_popularity(df_test, df_interacoes_treino, K_values=[5,10]):
    pop = df_interacoes_treino.groupby('produto')['contratou'].mean()
    pop_score = pop.to_dict()

    metrics_list = []
    for cliente_id, grp in df_test.groupby('id_cliente'):
        relevantes = set(grp[grp['contratou']==1]['produto'].values)
        if not relevantes:
            continue
        candidatos = grp['produto'].tolist()
        ranking = sorted(candidatos, key=lambda p: pop_score.get(p, 0), reverse=True)
        rel_scores = {p: 1 for p in relevantes}

        row = {'id_cliente': cliente_id}
        for k in K_values:
            row[f'precision@{k}'] = precision_at_k(ranking, relevantes, k)
            row[f'recall@{k}'] = recall_at_k(ranking, relevantes, k)
            row[f'hit_rate@{k}'] = hit_rate_at_k(ranking, relevantes, k)
            row[f'ndcg@{k}'] = ndcg_at_k(ranking, rel_scores, k)
            row[f'map@{k}'] = average_precision_at_k(ranking, relevantes, k)
        metrics_list.append(row)

    df_m = pd.DataFrame(metrics_list)
    print('\n=== Baseline: Popularidade ===')
    for k in K_values:
        print(f'\n  @{k}:')
        for m in ['precision','recall','hit_rate','ndcg','map']:
            col = f'{m}@{k}'
            print(f'    {col:<15}: {df_m[col].mean():.4f}')
    return df_m


def evaluate_baseline_random(df_test, K_values=[5,10], seed=42):
    rng = np.random.RandomState(seed)
    metrics_list = []
    for cliente_id, grp in df_test.groupby('id_cliente'):
        relevantes = set(grp[grp['contratou']==1]['produto'].values)
        if not relevantes:
            continue
        candidatos = grp['produto'].tolist()
        rng.shuffle(candidatos)
        rel_scores = {p: 1 for p in relevantes}

        row = {'id_cliente': cliente_id}
        for k in K_values:
            row[f'precision@{k}'] = precision_at_k(candidatos, relevantes, k)
            row[f'recall@{k}'] = recall_at_k(candidatos, relevantes, k)
            row[f'hit_rate@{k}'] = hit_rate_at_k(candidatos, relevantes, k)
            row[f'ndcg@{k}'] = ndcg_at_k(candidatos, rel_scores, k)
            row[f'map@{k}'] = average_precision_at_k(candidatos, relevantes, k)
        metrics_list.append(row)

    df_m = pd.DataFrame(metrics_list)
    print('\n=== Baseline: Aleatório ===')
    for k in K_values:
        print(f'\n  @{k}:')
        for m in ['precision','recall','hit_rate','ndcg','map']:
            col = f'{m}@{k}'
            print(f'    {col:<15}: {df_m[col].mean():.4f}')
    return df_m


metrics_pop = evaluate_baseline_popularity(df_test, int_treino)
metrics_rand = evaluate_baseline_random(df_test)

In [ ]:
# ── Comparativo visual ────────────────────────────────────────
K_values = [5, 10]
modelos = ['Aleatório', 'Popularidade', 'LambdaMART']
metric_names = ['precision', 'ndcg', 'hit_rate', 'map']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Comparativo de Modelos — Métricas de Recomendação', fontsize=14, fontweight='bold')

colors = ['#F44336', '#FF9800', '#2196F3']

for ki, k in enumerate(K_values):
    for mi, m in enumerate(metric_names):
        ax = axes[ki, mi]
        col = f'{m}@{k}'
        vals = [
            metrics_rand[col].mean(),
            metrics_pop[col].mean(),
            metrics_model[col].mean(),
        ]
        bars = ax.bar(modelos, vals, color=colors, edgecolor='white', linewidth=1.5)
        ax.set_title(f'{col.upper()}', fontweight='bold')
        ax.set_ylim(0, max(vals) * 1.3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
        ax.set_xticklabels(modelos, rotation=15, fontsize=9)

plt.tight_layout()
plt.savefig('comparativo_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Interpretabilidade — Exemplo de Recomendação

In [ ]:
def explicar_recomendacao(cliente_id, df_clientes, df_contratos, df_produtos,
                          cli_feat, hist_feat, pop_feat, pop_seg_feat,
                          model, FEATURE_COLS, le_cat, le_canal, top_k=5):
    """
    Gera e explica a recomendação para um cliente específico.
    """
    produtos_list = df_produtos['produto'].tolist()

    # Produtos ativos (excluir)
    ativos = set(
        df_contratos[
            (df_contratos['id_cliente']==cliente_id) &
            (df_contratos['status']=='ativo')
        ]['produto'].values
    )
    elegiveis = [p for p in produtos_list if p not in ativos]

    # Constrói features para (cliente, produto) elegível
    cliente_info = cli_feat[cli_feat['id_cliente']==cliente_id].iloc[0]

    rows = []
    for prod in elegiveis:
        row = {
            'id_cliente': cliente_id,
            'produto': prod,
            'posicao_exibicao': 1,  # posição hipotética
            'canal': 'app_home',
        }
        # Features do cliente
        for col in ['segmento_enc','canal_preferencial_enc','genero_enc','uf_freq',
                    'renda_log','saldo_log','inv_log','gasto_cartao_log','limite_log',
                    'renda_saldo_ratio','investimento_renda_ratio','gasto_total',
                    'propensao_credito','propensao_investimento','faixa_idade',
                    'score_credito','qtd_meses_cliente','qtd_produtos_ativos',
                    'qtd_transacoes_pix_6m','ind_debito_automatico','qtd_dias_inatividade',
                    'pct_utilizacao_limite']:
            row[col] = cliente_info.get(col, 0)

        # Histórico
        h = hist_feat[(hist_feat['id_cliente']==cliente_id) & (hist_feat['produto']==prod)]
        row['n_impressoes'] = h['n_impressoes'].values[0] if len(h) else 0
        row['n_cliques'] = h['n_cliques'].values[0] if len(h) else 0
        row['n_contratos'] = h['n_contratos'].values[0] if len(h) else 0
        row['ctr_hist'] = h['ctr_hist'].values[0] if len(h) else 0
        row['conv_hist'] = h['conv_hist'].values[0] if len(h) else 0
        row['pos_media'] = h['pos_media'].values[0] if len(h) else 10
        row['pos_min'] = h['pos_min'].values[0] if len(h) else 20
        row['receita_hist'] = h['receita_hist'].values[0] if len(h) else 0

        # Popularidade
        p = pop_feat[pop_feat['produto']==prod]
        row['pop_ctr'] = p['pop_ctr'].values[0] if len(p) else 0
        row['pop_conv'] = p['pop_conv'].values[0] if len(p) else 0
        row['pop_receita_media'] = p['pop_receita_media'].values[0] if len(p) else 0

        # Popularidade por segmento
        seg = cliente_info.get('segmento', 'basico')
        ps = pop_seg_feat[(pop_seg_feat['segmento']==seg) & (pop_seg_feat['produto']==prod)]
        row['seg_conv'] = ps['seg_conv'].values[0] if len(ps) else 0
        row['seg_ctr'] = ps['seg_ctr'].values[0] if len(ps) else 0

        # Produto
        prod_info = df_produtos[df_produtos['produto']==prod].iloc[0]
        row['categoria_enc'] = le_cat.transform([prod_info['categoria']])[0]
        row['receita_media'] = prod_info['receita_media']
        row['margem'] = prod_info['margem']

        row['canal_enc'] = le_canal.transform(['app_home'])[0]
        row['pos_bias'] = 1.0
        row['is_top5'] = 1

        rows.append(row)

    df_pred = pd.DataFrame(rows)
    X_pred = df_pred[FEATURE_COLS].fillna(0)
    df_pred['score'] = model.predict(X_pred)
    df_pred = df_pred.sort_values('score', ascending=False)

    print(f'\n====== Recomendações para Cliente {cliente_id} ======')
    print(f'Segmento: {cliente_info["segmento"]} | Renda: R$ {np.expm1(cliente_info["renda_log"]):,.0f}')
    print(f'Score Crédito: {cliente_info["score_credito"]:.0f} | '
          f'Investimentos: R$ {np.expm1(cliente_info["inv_log"]):,.0f}')
    print(f'Produtos ativos: {ativos or "nenhum"}')
    print(f'\nTop {top_k} recomendações:')
    print(f'{"Pos":<5} {"Produto":<25} {"Score":<10} {"Pop.Conv":<12} {"SegConv":<12} {"HistConv"}')
    print('-' * 75)
    for i, (_, row) in enumerate(df_pred.head(top_k).iterrows(), 1):
        print(f'{i:<5} {row["produto"]:<25} {row["score"]:<10.4f} '
              f'{row["pop_conv"]:<12.4f} {row["seg_conv"]:<12.4f} {row["conv_hist"]:.4f}')

    return df_pred


# Exemplo com 3 clientes de diferentes segmentos
for seg in ['basico', 'intermediario', 'premium']:
    ex_id = df_clientes[df_clientes['segmento']==seg]['id_cliente'].iloc[0]
    explicar_recomendacao(
        ex_id, df_clientes, df_contratos, df_produtos,
        cli_feat, hist_feat, pop_feat, pop_seg_feat,
        model, FEATURE_COLS, le_cat, le_canal
    )

## 8. Geração do arquivo recomendacoes.csv

In [ ]:
def gerar_recomendacoes_full(df_clientes, df_contratos, df_produtos,
                              cli_feat, hist_feat, pop_feat, pop_seg_feat,
                              model, FEATURE_COLS, le_cat, le_canal,
                              output_path='recomendacoes.csv',
                              batch_size=1000):
    """
    Gera rankings para TODOS os clientes, processando em batches.
    """
    produtos_list = df_produtos['produto'].tolist()
    prod_info_dict = df_produtos.set_index('produto').to_dict('index')

    # Produtos ativos por cliente
    ativos_dict = df_contratos[df_contratos['status']=='ativo'].groupby(
        'id_cliente'
    )['produto'].apply(set).to_dict()

    # Histórico indexado
    hist_idx = hist_feat.set_index(['id_cliente','produto'])
    pop_idx = pop_feat.set_index('produto')
    pop_seg_idx = pop_seg_feat.set_index(['segmento','produto'])

    all_recs = []
    ids = df_clientes['id_cliente'].values
    n_total = len(ids)

    for batch_start in range(0, n_total, batch_size):
        batch_ids = ids[batch_start:batch_start+batch_size]
        rows = []

        for cliente_id in batch_ids:
            ativos = ativos_dict.get(cliente_id, set())
            elegiveis = [p for p in produtos_list if p not in ativos]

            try:
                cli_row = cli_feat[cli_feat['id_cliente']==cliente_id].iloc[0]
                seg = cli_row.get('segmento', 'basico')
            except:
                continue

            for prod in elegiveis:
                row = {
                    '_cliente_id': cliente_id,
                    '_produto': prod,
                }

                # Features do cliente
                for col in ['segmento_enc','canal_preferencial_enc','genero_enc','uf_freq',
                            'renda_log','saldo_log','inv_log','gasto_cartao_log','limite_log',
                            'renda_saldo_ratio','investimento_renda_ratio','gasto_total',
                            'propensao_credito','propensao_investimento','faixa_idade',
                            'score_credito','qtd_meses_cliente','qtd_produtos_ativos',
                            'qtd_transacoes_pix_6m','ind_debito_automatico',
                            'qtd_dias_inatividade','pct_utilizacao_limite']:
                    row[col] = cli_row.get(col, 0)

                # Histórico
                try:
                    h = hist_idx.loc[(cliente_id, prod)]
                    row['n_impressoes'] = h['n_impressoes']
                    row['n_cliques'] = h['n_cliques']
                    row['n_contratos'] = h['n_contratos']
                    row['ctr_hist'] = h['ctr_hist']
                    row['conv_hist'] = h['conv_hist']
                    row['pos_media'] = h['pos_media']
                    row['pos_min'] = h['pos_min']
                    row['receita_hist'] = h['receita_hist']
                except:
                    for c in ['n_impressoes','n_cliques','n_contratos','ctr_hist',
                              'conv_hist','pos_media','receita_hist']:
                        row[c] = 0
                    row['pos_min'] = 20

                # Popularidade
                try:
                    p = pop_idx.loc[prod]
                    row['pop_ctr'] = p['pop_ctr']
                    row['pop_conv'] = p['pop_conv']
                    row['pop_receita_media'] = p['pop_receita_media']
                except:
                    row['pop_ctr'] = row['pop_conv'] = row['pop_receita_media'] = 0

                # Popularidade por segmento
                try:
                    ps = pop_seg_idx.loc[(seg, prod)]
                    row['seg_conv'] = ps['seg_conv']
                    row['seg_ctr'] = ps['seg_ctr']
                except:
                    row['seg_conv'] = row['seg_ctr'] = 0

                # Produto
                pi = prod_info_dict.get(prod, {})
                row['categoria_enc'] = le_cat.transform([pi.get('categoria','cartao')])[0]
                row['receita_media'] = pi.get('receita_media', 0)
                row['margem'] = pi.get('margem', 0)

                row['canal_enc'] = le_canal.transform(['app_home'])[0]
                row['pos_bias'] = 1.0
                row['is_top5'] = 1

                rows.append(row)

        if not rows:
            continue

        df_batch = pd.DataFrame(rows)
        X_batch = df_batch[FEATURE_COLS].fillna(0)
        df_batch['score'] = model.predict(X_batch)

        # Gerar rankings por cliente
        df_batch = df_batch.sort_values(['_cliente_id','score'], ascending=[True,False])
        df_batch['posicao'] = df_batch.groupby('_cliente_id').cumcount() + 1

        recs = df_batch[['_cliente_id','_produto','posicao','score']].rename(columns={
            '_cliente_id': 'id_cliente',
            '_produto': 'produto',
        })
        all_recs.append(recs)

        if (batch_start // batch_size) % 10 == 0:
            pct = min(batch_start + batch_size, n_total) / n_total * 100
            print(f'  Progresso: {pct:.0f}% ({min(batch_start+batch_size, n_total):,}/{n_total:,} clientes)')

    df_final = pd.concat(all_recs, ignore_index=True)
    df_final['score'] = df_final['score'].round(6)
    df_final.to_csv(output_path, index=False)

    print(f'\nrecomendacoes.csv gerado: {len(df_final):,} linhas')
    print(f'Clientes únicos: {df_final["id_cliente"].nunique():,}')
    print(df_final.head(10))
    return df_final


print('Gerando recomendações para todos os clientes...')
df_recs = gerar_recomendacoes_full(
    df_clientes, df_contratos, df_produtos,
    cli_feat, hist_feat, pop_feat, pop_seg_feat,
    model, FEATURE_COLS, le_cat, le_canal,
)

In [ ]:
# Validação do arquivo de saída
print('=== Validação do recomendacoes.csv ===')
print(f'Linhas totais: {len(df_recs):,}')
print(f'Clientes únicos: {df_recs["id_cliente"].nunique():,}')
print(f'Produtos únicos: {df_recs["produto"].nunique():,}')
print(f'\nDistribuição de posições por cliente:')
prods_por_cli = df_recs.groupby('id_cliente')['produto'].count()
print(prods_por_cli.describe())

# Verificar que produtos ativos não estão nas recomendações
ativos_set = set(zip(
    df_contratos[df_contratos['status']=='ativo']['id_cliente'],
    df_contratos[df_contratos['status']=='ativo']['produto']
))
recs_set = set(zip(df_recs['id_cliente'], df_recs['produto']))
overlap = ativos_set & recs_set
print(f'\nPares (cliente, produto ativo) nas recomendações: {len(overlap)} (deve ser 0)')

## 9. Proposta de Produção

In [ ]:
proposta_producao = """
╔══════════════════════════════════════════════════════════════════════════════╗
║                    PROPOSTA DE ARQUITETURA EM PRODUÇÃO                     ║
╚══════════════════════════════════════════════════════════════════════════════╝

1. ESTRATÉGIA: BATCH + CACHE (preferível para este caso)
   ─────────────────────────────────────────────────────
   • Recomendações geradas diariamente (job noturno) para todos os 50k clientes
   • Resultados armazenados em cache (Redis / DynamoDB) por id_cliente
   • API REST serve rankings do cache com latência <5ms
   • Justificativa: features de cliente mudam pouco intraday; batch reduz custo
     computacional vs inference em tempo real para 50k usuários

2. PIPELINE DE RETREINAMENTO
   ──────────────────────────
   • Frequência: mensal (safra completa)
   • Trigger automático: queda de NDCG@5 > 5% em monitoramento contínuo
   • Split: sempre temporal — últimas 3 safras como validação
   • Versão do modelo salva com MLflow; rollback automático se métricas pioram

3. MÉTRICAS DE MONITORAMENTO EM PRODUÇÃO
   ────────────────────────────────────────
   Online (A/B test + logs):
     • CTR nas 5 primeiras posições (vs controle estático)
     • Taxa de contratação por posição e produto
     • Receita gerada por sessão
     • Coverage: % de clientes recebendo recomendações
   Offline (batch):
     • NDCG@5 e Precision@5 na safra mais recente
     • Desvio de distribuição de features (PSI > 0.2 = alerta)
     • Latência de serving e disponibilidade do cache

4. DETECÇÃO DE DEGRADAÇÃO
   ─────────────────────────
   • Data drift: PSI (Population Stability Index) em features-chave
   • Model drift: comparação rolling de NDCG@5 (últimas 2 vs 4 semanas)
   • Business drift: queda de CTR ou conversão > 10% vs baseline histórico
   • Alertas automáticos via Slack/PagerDuty com threshold configurável

5. EXPERIMENTOS E EVOLUÇÃO
   ─────────────────────────
   • A/B test: 90% modelo v1 / 10% nova versão; promoção automática se p-value < 0.05
   • Exploração: Thompson Sampling em produtos com poucos dados (cold start)
   • Enriquecimento: adicionar features de comportamento em tempo real
     (ex: cliques na última hora) via feature store (Feast/Tecton)
"""
print(proposta_producao)

## 10. Sumário Final de Métricas

In [ ]:
print('\n' + '='*65)
print('SUMÁRIO FINAL DE MÉTRICAS')
print('='*65)

header = f'{"Modelo":<20} {"P@5":>8} {"NDCG@5":>8} {"HR@5":>8} {"MAP@5":>8}'
print(header)
print('-'*65)

for label, m in [('Aleatório', metrics_rand), ('Popularidade', metrics_pop), ('LambdaMART', metrics_model)]:
    print(f'{label:<20} {m["precision@5"].mean():>8.4f} {m["ndcg@5"].mean():>8.4f} '
          f'{m["hit_rate@5"].mean():>8.4f} {m["map@5"].mean():>8.4f}')

print('='*65)

# Ganho relativo
for metric in ['precision@5', 'ndcg@5', 'hit_rate@5']:
    ganho_vs_rand = (metrics_model[metric].mean() / metrics_rand[metric].mean() - 1) * 100
    ganho_vs_pop = (metrics_model[metric].mean() / metrics_pop[metric].mean() - 1) * 100
    print(f'{metric}: +{ganho_vs_rand:.1f}% vs Aleatório | +{ganho_vs_pop:.1f}% vs Popularidade')

print(f'\nModelo salvo. Arquivo recomendacoes.csv gerado com sucesso.')
print('Análise concluída!')

---
## Limitações e Próximos Passos

**Limitações:**
- Dados sintéticos: distribuições podem não capturar padrões reais de comportamento bancário
- Cold start: clientes sem histórico de interação têm scores baseados apenas em popularidade por segmento
- Feedback positivo-only: só observamos interações (impressão), não a ausência de interesse real
- Position bias: embora incluído como feature, o modelo pode subestimar conversões em posições baixas

**Próximos Passos:**
1. Incorporar embeddings de produto (Word2Vec sobre sequências de contratação)
2. Adicionar features de sazonalidade (mês, dia da semana, fim de mês)
3. Testar Two-Tower Neural Network para melhor representação de usuário/produto
4. Implementar Contextual Bandit para exploração ativa de novos produtos
5. Calibração de scores para conversão direta em probabilidade monetária esperada